# Extra: A Transformer That Is a Turing Machine

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/A2-Transformer-Turing-Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_tm_program.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In A1 we hand-designed a Transformer that executes a small CPU. Here we go to the most fundamental model of computation, the **Turing machine**, and design a Transformer that executes **any** (small) Turing machine, exactly, with weights written by hand. Then we look at what this says about the theory: is a Transformer "Turing complete"?

The notebook is mostly explanation; the code is small and runs in seconds on a CPU, a T4 or a TPU.

In [ ]:
import random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

## 1. Turing machines

A Turing machine (Alan Turing, 1936) has

* an infinite **tape** of cells, each holding one symbol from a finite alphabet (here `_` = blank, `0`, `1`);
* a **head** that sits on one cell and can read and write it;
* a finite set of **states**, one of which is the current state, plus a special **halt** state;
* a **transition table** $\delta$: for every (state, symbol under the head) it says
  1. which symbol to **write**,
  2. whether to **move** the head Left, Stay or Right,
  3. which state to go to **next**.

That is all. One step: read, look up $\delta$, write, move, change state. Repeat until the halt state.
Despite this simplicity, a Turing machine can compute anything any computer can (the **Church–Turing thesis**).

A **universal** Turing machine is one whose *input* contains the description (the table) of another machine, and which then simulates it. The program is data. This is exactly the idea of a stored-program computer, and it is what we will build: one Transformer, with fixed weights, that runs whatever transition table we give it as input.

Our machine has a finite capacity (real hardware always does):

In [ ]:
# ---------------- capacity of the universal machine ----------------
N_STATES = 6            # including the halting state (index N_STATES - 1)
SYMBOLS = "_01"         # "_" is the blank symbol
N_SYM = len(SYMBOLS)
TAPE = 32               # tape cells
MOVES = "LSR"           # left, stay, right
HALT = N_STATES - 1
N_ENTRIES = N_STATES * N_SYM

### A reference simulator

The plain Python version, to check the Transformer against. A table is a dict `(state, symbol) -> (write, move, next)`.
At the edges of our finite tape the head simply stays where it is.

In [ ]:
def tm_step(table, state, pos, tape):
    """table[(q, s)] = (write, move, next).  Returns the new (state, pos, tape)."""
    w, m, nxt = table[(state, tape[pos])]
    tape = tape[:pos] + [w] + tape[pos + 1:]
    pos = min(max(pos + {"L": -1, "S": 0, "R": 1}[m], 0), TAPE - 1)
    return nxt, pos, tape

def complete(table):
    """Fill in the missing entries: unknown -> halt without changing anything; the HALT state does nothing."""
    full = {}
    for q in range(N_STATES):
        for s in range(N_SYM):
            full[(q, s)] = table.get((q, s), (s, "S", HALT)) if q != HALT else (s, "S", HALT)
    return full

In [ ]:
def machine(spec, state_names):
    """Write a machine compactly: {"A_": "1RB", ...} means: in state A reading "_", write 1, move Right, go to B.
    State H is the halt state."""
    index = {n: i for i, n in enumerate(state_names)}
    index["H"] = HALT
    return complete({(index[k[0]], SYMBOLS.index(k[1])): (SYMBOLS.index(v[0]), v[1], index[v[2]])
                     for k, v in spec.items()})

def show(state, pos, tape, state_names="ABCDE"):
    cells = "".join(SYMBOLS[s] for s in tape)
    name = "H" if state == HALT else state_names[state]
    return cells[:pos] + "[" + cells[pos] + "]" + cells[pos + 1:] + "  state " + name

# Binary increment: start on the last digit of a binary number, in state A ("carry").
INC = machine({"A1": "0LA",      # 1 + carry = 0, carry continues to the left
               "A0": "1SH",      # 0 + carry = 1, done
               "A_": "1SH"},     # past the leftmost digit: write a new 1, done
              "A")
tape = [0] * TAPE
for i, c in enumerate("1011"):
    tape[10 + i] = SYMBOLS.index(c)
state, pos = 0, 13
print(show(state, pos, tape))
while state != HALT:
    state, pos, tape = tm_step(INC, state, pos, tape)
    print(show(state, pos, tape))

## 2. The design

### 2.1 Tokens and the residual stream

The Transformer sees a sequence of $18 + 1 + 32 = 51$ tokens:

| Tokens | How many | What their residual vectors contain |
|---|---|---|
| **table** tokens | one per (state, symbol) pair: $6 \times 3 = 18$ | one-hot `tkey` of the pair, and the entry: one-hot `twrite`, `tmove`, `tnext` |
| the **head** token | 1 | one-hot current `state`, one-hot head position `hpos` |
| **tape** tokens | one per cell: 32 | one-hot own position `tpos`, one-hot symbol `sym` |

As in A1, every dimension of the residual stream has a **name and a meaning** that we choose, and every value is 0 or 1. A type flag (`is_table`, `is_head`, `is_tape`) says what kind of token it is. Extra **scratch** dimensions hold intermediate results.

### 2.2 Attention = an exact table lookup

Attention computes, for each token, a weighted average of the other tokens' **values**, with weights $\mathrm{softmax}(\beta\, q \cdot k)$.
If the query $q$ is a one-hot vector and every key $k$ is a one-hot vector, then $q \cdot k = 1$ for the one token whose key matches and $0$ for all others. With a large scale $\beta$ (we use 1000):

$$\mathrm{softmax}(\beta\,q\cdot k)_j = \frac{e^{\beta [j \text{ matches}]}}{\sum_{j'} e^{\beta [j' \text{ matches}]}} \approx \begin{cases}1 & j \text{ matches}\\ e^{-1000} = 0 & \text{otherwise}\end{cases}$$

so attention returns **exactly the value of the matching token**. Attention is a **content-addressable memory**: "give me the value stored under this key". A Turing machine needs exactly two lookups (the symbol under the head, the table entry for (state, symbol)), plus one broadcast (tell every tape cell what the head decided).

### 2.3 MLPs = logic gates

For $a, b, c \in \{0, 1\}$, one ReLU neuron computes AND exactly:

$$\mathrm{ReLU}(a + b - 1) = a \wedge b, \qquad \mathrm{ReLU}(a + b + c - 2) = a \wedge b \wedge c$$

and, through the residual connection, the MLP can **add** $+1$ or $-1$ to any dimension. "Set bit $x$ to $y$ if $c$" becomes "add $c \wedge y$, subtract $c \wedge x$".

These gates also keep the computation clean: on tokens where a gate input should be 0 (say, the "is this the head token" flag on a tape token), the AND is 0 whatever the other inputs contain. That is why we never have to worry about what attention writes into the scratch dimensions of tokens we don't care about.

In [ ]:
def plus(*ds):
    out = defaultdict(float)
    for d in ds:
        for k, v in d.items():
            out[k] += v
    return dict(out)

def and_(layer, a, b, out):
    """out += a AND b           (a, b in {0, 1}):  ReLU(a + b - 1)"""
    layer.append((plus(a, b), -1, out))

def and3(layer, a, b, c, out):
    """out += a AND b AND c     (a, b, c in {0, 1}):  ReLU(a + b + c - 2)"""
    layer.append((plus(a, b, c), -2, out))

one = lambda name: {name: 1}

### 2.4 Three blocks = one step of the machine

```
                       table tokens  ──────────────┐
                                                   │ (2) LOOK UP δ(state, symbol)
 tape tokens ─(1) READ the symbol under the head─▶ HEAD ──(3) BROADCAST write + position──▶ tape tokens
                                                   │
                                     MLP: new state, move the head        MLP: write the cell under the head
```

**Block 1 — read.** Attention: the head token's query is its one-hot position `hpos`, the tape tokens' keys are their one-hot positions `tpos`, so the head token receives the one-hot symbol under the head (`read`). MLP: `pair[q·3+s] = state[q] AND read[s]`, the one-hot of the pair (state, symbol), i.e. the address of the table entry to use.

In [ ]:
blocks = []

# Block 1 -- READ: the head token looks up the tape cell at its own position and copies the symbol.
read = dict(query=[f"hpos{i}" for i in range(TAPE)], key=[f"tpos{i}" for i in range(TAPE)],
            value=[f"sym{s}" for s in range(N_SYM)], out=[f"read{s}" for s in range(N_SYM)])
mlp1 = []
for q in range(N_STATES):                       # one-hot of the pair (state, symbol read)
    for s in range(N_SYM):
        and_(mlp1, one(f"state{q}"), one(f"read{s}"), one(f"pair{q * N_SYM + s}"))
blocks.append((read, mlp1))

**Block 2 — look up the transition.** Attention: query `pair`, keys `tkey` of the 18 table tokens. The head token receives `write`, `move` and `next` of the matching entry. No MLP needed.

In [ ]:
# Block 2 -- LOOK UP the transition: the head token finds the table token for its (state, symbol)
lookup = dict(query=[f"pair{k}" for k in range(N_ENTRIES)], key=[f"tkey{k}" for k in range(N_ENTRIES)],
              value=[f"twrite{s}" for s in range(N_SYM)] + [f"tmove{m}" for m in range(3)] +
                    [f"tnext{q}" for q in range(N_STATES)],
              out=[f"write{s}" for s in range(N_SYM)] + [f"move{m}" for m in range(3)] +
                  [f"next{q}" for q in range(N_STATES)])
blocks.append((lookup, []))

**Block 3 — broadcast and update.** Attention: every tape token has query `is_tape` = 1 and the head token has key `is_head` = 1, so every tape token receives a copy of the head's position (`hp`) and of the symbol to write (`hw`). Then one MLP does all the updates at once:

* **tape**: in cell $i$, if `tpos[i] AND hp[i]` (the head is here), add `hw[s]` and subtract the old `sym[s]`: the symbol is replaced. In all other cells nothing changes.
* **state**: on the head token, add `next[q]` and subtract the old `state[q]`.
* **head position**: the new one-hot position is the old one shifted: `new[i] = (move=S AND old[i]) + (move=R AND old[i-1]) + (move=L AND old[i+1])` (and the head stays put at the edges).

Writing the cell and moving the head happen in the same MLP, but both use the head position **from before** the move, because the MLP's inputs are read before its outputs are added: the order "write, then move" comes for free.

In [ ]:
# Block 3 -- BROADCAST + UPDATE: every tape token copies the head position and the symbol to write;
#            the MLP writes the tape cell under the head, changes the state and moves the head.
broadcast = dict(query=["is_tape"], key=["is_head"],
                 value=[f"write{s}" for s in range(N_SYM)] + [f"hpos{i}" for i in range(TAPE)],
                 out=[f"hw{s}" for s in range(N_SYM)] + [f"hp{i}" for i in range(TAPE)])
mlp3 = []
for i in range(TAPE):                           # tape: if the head is here, replace the symbol
    for s in range(N_SYM):
        and3(mlp3, one(f"tpos{i}"), one(f"hp{i}"), one(f"hw{s}"), {f"sym{s}": +1})
        and3(mlp3, one(f"tpos{i}"), one(f"hp{i}"), one(f"sym{s}"), {f"sym{s}": -1})
for q in range(N_STATES):                       # head token: state <- next
    and_(mlp3, one("is_head"), one(f"next{q}"), {f"state{q}": +1})
    and_(mlp3, one("is_head"), one(f"state{q}"), {f"state{q}": -1})
L, S, R = (one(f"move{m}") for m in range(3))
for i in range(TAPE):                           # head token: shift the one-hot head position
    sources = [(S, i)]
    if i > 0:
        sources.append((R, i - 1))              # moving right from i-1 lands on i
    if i < TAPE - 1:
        sources.append((L, i + 1))              # moving left from i+1 lands on i
    if i == 0:
        sources.append((L, 0))                  # the head cannot leave the tape
    if i == TAPE - 1:
        sources.append((R, TAPE - 1))
    for move, j in sources:
        and3(mlp3, one("is_head"), move, one(f"hpos{j}"), {f"hpos{i}": +1})
    and_(mlp3, one("is_head"), one(f"hpos{i}"), {f"hpos{i}": -1})
blocks.append((broadcast, mlp3))

### 2.5 Compiling into a Transformer, and the loop

As in A1, each named dimension gets an index, and every lookup and neuron is written into ordinary `nn.Linear` weights; the forward pass uses standard `scaled_dot_product_attention`.

One forward pass is **one step** of the machine. To run the machine we feed the output back in as the next input: a **looped Transformer** (the depth comes from repeating, not from more layers). Before feeding it back we clear the scratch dimensions, so the next step starts from a clean residual stream containing only the machine's state, exactly like an embedding of the new configuration.

In [ ]:
names = {}
def dim(name):
    return names.setdefault(name, len(names))

for name in (["is_table", "is_head", "is_tape"] + [f"tkey{k}" for k in range(N_ENTRIES)] +
             [f"twrite{s}" for s in range(N_SYM)] + [f"tmove{m}" for m in range(3)] +
             [f"tnext{q}" for q in range(N_STATES)] + [f"state{q}" for q in range(N_STATES)] +
             [f"hpos{i}" for i in range(TAPE)] + [f"tpos{i}" for i in range(TAPE)] +
             [f"sym{s}" for s in range(N_SYM)]):
    dim(name)
N_INPUT_DIMS = len(names)        # dims 0..N_INPUT_DIMS-1 hold the machine; the rest is scratch space
for attn, mlp in blocks:
    for key in ("query", "key", "value", "out"):
        for name in attn[key]:
            dim(name)
    for inp, b, out in mlp:
        for name in list(inp) + list(out):
            dim(name)
D_MODEL = len(names)


class DesignedBlock(nn.Module):
    """x = x + Attention(x);  x = x + MLP(x)   (one head, no LayerNorm, weights set by hand)"""
    def __init__(self, attn, mlp, beta=1000.0):
        super().__init__()
        self.beta = beta
        self.Wq = nn.Linear(D_MODEL, len(attn["query"]), bias=False)
        self.Wk = nn.Linear(D_MODEL, len(attn["key"]), bias=False)
        self.Wv = nn.Linear(D_MODEL, len(attn["value"]), bias=False)
        self.Wo = nn.Linear(len(attn["value"]), D_MODEL, bias=False)
        for W in (self.Wq, self.Wk, self.Wv, self.Wo):
            nn.init.zeros_(W.weight)
        for i, (q, k) in enumerate(zip(attn["query"], attn["key"])):
            self.Wq.weight.data[i, names[q]] = 1.0
            self.Wk.weight.data[i, names[k]] = 1.0
        for i, (v, o) in enumerate(zip(attn["value"], attn["out"])):
            self.Wv.weight.data[i, names[v]] = 1.0
            self.Wo.weight.data[names[o], i] = 1.0
        self.mlp = None
        if mlp:
            self.fc1 = nn.Linear(D_MODEL, len(mlp))
            self.fc2 = nn.Linear(len(mlp), D_MODEL, bias=False)
            nn.init.zeros_(self.fc1.weight); nn.init.zeros_(self.fc2.weight)
            for h, (inp, b, out) in enumerate(mlp):
                for name, w in inp.items():
                    self.fc1.weight.data[h, names[name]] = w
                self.fc1.bias.data[h] = b
                for name, w in out.items():
                    self.fc2.weight.data[names[name], h] = w
            self.mlp = True

    def forward(self, x):
        q, k, v = self.Wq(x), self.Wk(x), self.Wv(x)
        x = x + self.Wo(F.scaled_dot_product_attention(q, k, v, scale=self.beta))
        if self.mlp:
            x = x + self.fc2(F.relu(self.fc1(x)))
        return x


class TransformerTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([DesignedBlock(a, m) for a, m in blocks])
        keep = torch.zeros(D_MODEL)
        keep[:N_INPUT_DIMS] = 1          # the machine's state; everything after is scratch space
        self.register_buffer("keep", keep)

    def forward(self, x):
        """One Turing-machine step. The output, with the scratch dimensions cleared, is the next input."""
        for blk in self.blocks:
            x = blk(x)
        return x * self.keep

In [ ]:
def encode(table, state, pos, tape):
    """One sequence of tokens: [table entries ..., head, tape cells ...]"""
    x = torch.zeros(N_ENTRIES + 1 + TAPE, D_MODEL)
    for k in range(N_ENTRIES):
        q, s = divmod(k, N_SYM)
        w, m, nxt = table[(q, s)]
        for name in ["is_table", f"tkey{k}", f"twrite{w}", f"tmove{MOVES.index(m)}", f"tnext{nxt}"]:
            x[k, names[name]] = 1
    h = N_ENTRIES
    for name in ["is_head", f"state{state}", f"hpos{pos}"]:
        x[h, names[name]] = 1
    for i, s in enumerate(tape):
        for name in ["is_tape", f"tpos{i}", f"sym{s}"]:
            x[h + 1 + i, names[name]] = 1
    return x

def decode(x):
    """Read (state, head position, tape) back from the residual stream"""
    h = N_ENTRIES
    state = x[h, [names[f"state{q}"] for q in range(N_STATES)]].argmax().item()
    pos = x[h, [names[f"hpos{i}"] for i in range(TAPE)]].argmax().item()
    sym = x[h + 1:, [names[f"sym{s}"] for s in range(N_SYM)]].argmax(1).tolist()
    return state, pos, sym

In [ ]:
tm = TransformerTM().to(device)
n_params = sum(p.numel() for p in tm.parameters())
n_nonzero = sum((p != 0).sum().item() for p in tm.parameters())
print(f"d_model = {D_MODEL} ({N_INPUT_DIMS} for the machine, {D_MODEL - N_INPUT_DIMS} scratch), "
      f"MLP widths = {[len(m) for _, m in blocks]}")
print(f"parameters: {n_params:,}, of which non-zero: {n_nonzero:,}")

## 3. Looking inside one step

Take the increment machine at its first step and look at where the head token attends in block 1 (which tape cell?) and in block 2 (which table entry?):

In [ ]:
tape = [0] * TAPE
for i, c in enumerate("1011"):
    tape[10 + i] = SYMBOLS.index(c)
x = encode(INC, 0, 13, tape)[None].to(device)
HEAD = N_ENTRIES
with torch.no_grad():
    for b, blk in enumerate(tm.blocks[:2]):
        q, k = blk.Wq(x), blk.Wk(x)
        weights = torch.softmax(blk.beta * q[0, HEAD] @ k[0].T, dim=-1)
        j = weights.argmax().item()
        if j >= HEAD + 1:
            where = f"tape cell {j - HEAD - 1} (symbol {SYMBOLS[tape[j - HEAD - 1]]!r})"
        else:
            st, sy = divmod(j, N_SYM)
            where = f"table entry (state {st}, symbol {SYMBOLS[sy]!r}) -> {INC[(st, sy)]}"
        print(f"block {b + 1}: the head token puts weight {weights[j].item():.6f} on {where}")
        x = blk(x)

## 4. Is it exact?

Compare with the reference simulator on 300 random machines (random tables, tapes, states and head positions), 50 steps each, checking the whole configuration after every step:

In [ ]:
def run_transformer(table, state, pos, tape, steps):
    x = encode(table, state, pos, tape)[None].to(device)
    with torch.no_grad():
        for _ in range(steps):
            x = tm(x)
            yield decode(x[0].cpu())

random.seed(0)
checked = mismatches = 0
for trial in range(300):
    table = complete({(q, s): (random.randrange(N_SYM), random.choice(MOVES), random.randrange(N_STATES))
                      for q in range(N_STATES - 1) for s in range(N_SYM)})
    state, pos = random.randrange(N_STATES - 1), random.randrange(TAPE)
    tape = [random.randrange(N_SYM) for _ in range(TAPE)]
    for got in run_transformer(table, state, pos, tape, 50):
        state, pos, tape = tm_step(table, state, pos, tape)
        checked += 1
        mismatches += got != (state, pos, tape)
print(f"{checked} steps checked, {mismatches} mismatches")

## 5. Running real machines

**Busy beaver.** Among all 4-state, 2-symbol machines that eventually halt when started on a blank tape, which one runs the longest? The answer (Brady, 1983) runs for **107 steps** and leaves **13 ones**. Our Transformer, with the same weights as before, only a different table as input:

In [ ]:
BB4 = machine({"A_": "1RB", "A1": "1LB", "B_": "1LA", "B1": "_LC",
               "C_": "1RH", "C1": "1LD", "D_": "1RD", "D1": "_RA"}, "ABCD")
state, pos, tape = 0, 16, [0] * TAPE
steps = 0
for state, pos, tape in run_transformer(BB4, state, pos, tape, 200):
    steps += 1
    if steps % 20 == 0 or state == HALT:
        print(f"{steps:4d}  {show(state, pos, tape, 'ABCD')}")
    if state == HALT:
        break
print(f"halted after {steps} steps with {tape.count(SYMBOLS.index('1'))} ones")

**A machine that never halts**: a binary counter. State A adds 1 (like the increment machine), state B walks back to the right end, and it repeats forever. Every time it returns to the right end, the tape shows the next number.

In [ ]:
COUNTER = machine({"A1": "0LA", "A0": "1RB", "A_": "1RB",       # add 1 with carry, then go right
                   "B0": "0RB", "B1": "1RB", "B_": "_LA"},      # walk to the right end, then add again
                  "AB")
tape = [0] * TAPE
tape[20] = SYMBOLS.index("0")
numbers = []
for state, pos, tape in run_transformer(COUNTER, 0, 20, tape, 400):
    if state == 0 and pos == 20:                    # back at the last digit: one more increment begins
        numbers.append(int("".join(SYMBOLS[s] for s in tape).strip("_"), 2))
print(numbers)

## Q
Write your own machine and run it on both the reference simulator and the Transformer. For example, **unary addition**: the tape `111_11` (3 + 2) should become `11111` (5).
Hint: walk right over the first number, turn the blank in the middle into `1`, walk to the end, step back and erase the last `1`.

In [ ]:
# %load q_tm_program.py
%run -i q_tm_program.py

## 6. What does this say about Transformers and computation?

**Our Transformer is a universal machine, but a finite one.** Its weights are fixed; the machine to run is input data (the table tokens), just as in a universal Turing machine. But the tape has 32 cells, there are at most 6 states and 3 symbols, and the head position is a one-hot vector of length 32. With a fixed number of tokens, any Transformer is a finite-state system, and finite-state systems are not Turing complete.

**Where would unbounded memory come from?** In a Transformer, memory is the **context**: one token per tape cell. If the model may **append tokens** (as an LLM does when it generates text), the context can grow without limit. This is the setting of the theoretical results:

* *Pérez, Barceló & Marinković (2019, 2021)*: Transformers with hard attention that keep generating tokens are Turing complete; each generated token records one step of the computation, which is essentially our design, with the tape written into the growing sequence.
* *Merrill & Sabharwal (2024)*: a Transformer that answers in one forward pass is limited to a small complexity class; allowing $T$ intermediate "chain of thought" tokens lets it simulate $T$ steps of a Turing machine. This is one theoretical reason why reasoning step by step helps LLMs.
* *Giannou et al. (2023), Looped Transformers*: a fixed Transformer applied in a loop (as we do) can run programs for a general-purpose instruction set.

Two technical points matter in those proofs: the head position must be representable for an **unbounded** tape (our one-hot vector of length 32 is not; positional encodings that grow with the sequence are), and **hard attention** must stay exact as the sequence grows.

**Comparison with RNNs/LSTMs.** An RNN keeps all of its memory in one fixed-size vector. The classic proof that RNNs are Turing complete (Siegelmann & Sontag, 1995) stores the whole tape in the *digits* of a few real numbers, which needs infinite precision; with float32, an LSTM is a finite-state machine. A Transformer instead keeps its memory as tokens and reads them with attention, so its memory grows with the context while each number stays small. That is a real architectural difference.

**Designed vs. trained.** As in A1, every weight here was written by hand, and the result is exact. A trained Transformer has no reason to find these clean circuits, and small per-step errors add up over long computations (A1, Part 2). This is why LLMs are much more reliable at long exact computations when they write them out step by step, or call a tool that really executes code.

## Further reading

* [Turing machine](https://en.wikipedia.org/wiki/Turing_machine) and [busy beaver](https://en.wikipedia.org/wiki/Busy_beaver) (Wikipedia); [bbchallenge.org](https://bbchallenge.org) for the collaborative proof that BB(5) = 47,176,870
* Pérez, Marinković & Barceló, [On the Turing Completeness of Modern Neural Network Architectures](https://arxiv.org/abs/1901.03429) (2019)
* Merrill & Sabharwal, [The Expressive Power of Transformers with Chain of Thought](https://arxiv.org/abs/2310.07923) (2024)
* Giannou et al., [Looped Transformers as Programmable Computers](https://arxiv.org/abs/2301.13196) (2023)
* Lindner et al., [Tracr: Compiled Transformers as a Laboratory for Interpretability](https://arxiv.org/abs/2301.05062) (2023)
* Siegelmann & Sontag, On the Computational Power of Neural Nets (1995): the classic RNN result